# PLL battery tests

Reference current (IREF) and PLL supply from batteries (`sw/tests/pll_test_vdd.py`, `vdd_sweep(vdds=None, ...)`), against the original IREF and the battery IREF with the PLL supply on the E36312A (0.75 V). IREF from a battery again after the reversed tantalums on the board were fixed (28 Oct 16:50),, the 28 Oct 16:25 run (the hump moved from 1.65 to ~3.9 MHz) and two runs with IREF and PLL supply from batteries: the cleanest before the fix (26 Oct 15:55) and one with new batteries (28 Oct 17:41).

Per setting (`pll_spur_zoom.SPUR_POINTS`): a summary table (lock, RMS jitter, L(f) at the decades), the Phase Noise mode L(f), the RMS jitter (10 kHz-10 MHz, `JITTER_BAND`, integrated from the Phase Noise mode L(f)) and the spectrum around the carrier, all referred to the VCO. Then the widest spectra (the 42 MHz span of the full measurement and, from new runs, the 1 MHz-1.5 GHz capture) and the jitter integrated over the offset (where it comes from). Figures in `results/pll/figures_battery/`.

## Storyline of the runs

Reconstructed from the result files and the lab notes in the chat; times are local, run = `pll_test_vdd_S5_<stamp>` in `results/pll/`. Jitter: the analyzer's Phase Noise mode result in the CSV (`time_jitter_pnm_ps`, 1 kHz-20 MHz); the plots and table below use 10 kHz-10 MHz instead. **Plotted** = in the run list of this notebook. Where the setup is not known for sure, the entry says so.

| Run | Setup | What happened | 760 MHz | 1 GHz | Plotted |
|---|---|---|---|---|---|
| 21 Oct 17:32 (`pll_spur_vdd_..._173215`) | original IREF, PLL supply swept | first look: the ~2 MHz hump and the spurs do not move with the supply, their level collapses below ~0.70 V | | | no |
| **21 Oct 18:13** (`_20261021_181301`) | original IREF, PLL supply on the E36312A, swept 0.60-0.85 V (0.75 V first and last) | 760 MHz jitter falls with the supply (359 ps at 0.75 V, 85 ps at 0.60 V); 1 GHz only locks at 0.75 V | 359 ps | 46 ps | yes (0.75 V rows), *IREF original* |
| **22 Oct 13:19** (`_20261022_131911`) | IREF from a battery, PLL supply on the bench (0.75 V) | best result for a long time | 101 ps | 14 ps | yes, *IREF battery* |
| 22 Oct 14:02 (`_20261022_140238`, spectra only) | as 13:19 | one 760 MHz capture, carrier at -11.6 dBm; no measurement. Around `pll_filter_tune` (14:20, 12.2 ps at 1 GHz; the PLL current jumped 0.22 mA between 13:22 and 14:22, cause unknown) | | | no |
| 26 Oct 12:16 (`_20261026_121628`, spectra only) | IREF + PLL supply from batteries (supply via LDOs) | 760 MHz not locked, stopped | | | no |
| 26 Oct 12:17 (`_20261026_121721`) | IREF + PLL supply from batteries | worse than 22 Oct ("worse again"); pressing the battery changed the spectrum, not the current | 167 ps | not locked | no |
| 26 Oct 15:32 (`_20261026_153228`) | IREF + PLL supply from batteries | re-run, stopped after the 760 MHz spectra (no Phase Noise mode result) | | | no |
| 26 Oct 15:42 (`_20261026_154253`) | IREF + PLL supply from batteries | re-run | 176 ps | not locked | no |
| **26 Oct 15:55** (`_20261026_155550`) | IREF + PLL supply from batteries | re-run, the cleanest battery run before the tantalum fix (only one with both settings locked and a Phase Noise mode result) | 199 ps | 20 ps | yes, *IREF + supply battery* (orange) |
| 26 Oct 16:12 (`_20261026_161259`) | IREF + PLL supply from batteries, FPGA unplugged after every lock (`unplug_fpga=True`) | 760 MHz a bit better (lowest random part of the day) but a 1.95 MHz spur at -34 dBc; 1 GHz locked but carrier at -23 dBm, no result. Unplugging dropped afterwards | 151 ps | no result | no |
| 28 Oct 15:54 (`_20261028_155416`) | decoupling added on the board (tantalums turned out to be reversed) | huge phase noise, carrier 7-8 dB low; the spectra were taken but the full measurement found the PLL NOT LOCKED | not locked | not locked | no |
| 28 Oct 16:12 (`_20261028_161214`, spectra only) | as 15:54 | not locked, spectra of the noise floor (carrier -102 dBm) still taken; led to the software fix in `capture_spectra`: no spectra without lock | | | no |
| 28 Oct 16:17 (`_20261028_161749`) | as 15:54 | first run with the fix: neither setting locked, nothing captured | not locked | not locked | no |
| **28 Oct 16:25** (`_20261028_162507`) | supplies reattached ("i reattached the supplies"), still before the tantalum fix; whether the PLL supply was the battery or the bench was not noted | the 1.65 MHz hump is gone but a ~2 MHz wide hump sits at ~3.85 MHz (-98 dBc/Hz) instead, 93 % of the jitter power | not locked | 30 ps | yes, *hump at 3.9 MHz* |
| **28 Oct 16:50** (`_20261028_165040`) | reversed tantalums fixed; IREF from a battery, PLL supply **not** a battery | lowest jitter of all data: 11.0 ps at 1 GHz (10 kHz-10 MHz), 79 % of it in the 1-3 MHz hump (~5 ps without it) | 96 ps | 12 ps | yes, *IREF battery* (aqua) |
| 28 Oct 17:31 (`_20261028_173131`) | new batteries as PLL supply (same as 17:41) | aborted on an analyzer VISA timeout; 1 GHz carrier already ~0.7 dB lower | error | no result | no |
| **28 Oct 17:41** (`_20261028_174157`) | IREF + PLL supply from (new) batteries | 760 MHz best so far (85 ps), but 1 GHz ruined by a spur comb at multiples of 1.953 MHz (-11 dBc at 3.906 MHz): the battery supply hookup, not the PLL (26 Oct battery supply gave 20 ps) | 85 ps | 1151 ps | yes, *IREF + supply battery* (yellow) |

In [55]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise, split_jitter

FIG_DIR = PLL_DIR / "figures_battery"  # every figure as HTML + PNG
FIG_DIR.mkdir(exist_ok=True)

GRID = "#e4e3df"
FONT = dict(size=16)
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"), gridcolor=GRID)
N_BINS = 300  # points per plotted phase-noise curve (display only)
F_START, F_STOP = 1e3, 20e6  # offset range of the L(f) plots [Hz]
JITTER_BAND = (10e3, 10e6)  # RMS jitter integration band [Hz], integrated here from the Phase Noise mode L(f) (the
# analyzer's own time_jitter_pnm_ps in the CSV is 1 kHz-20 MHz)
BAND_TEXT = f"{JITTER_BAND[0] / 1e3:g} kHz-{JITTER_BAND[1] / 1e6:g} MHz"


def binned(df, f_start, f_stop):
    """Curve df (freq, level_dbm) between f_start and f_stop in N_BINS log-spaced bins (linear power mean)."""
    df = df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)].sort_values("freq")
    if len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name, width=1200):
    """HTML and PNG (kaleido) in FIG_DIR, then show the figure."""
    fig.update_layout(template="plotly_white", font=FONT)
    fig.write_html(FIG_DIR / f"{name}.html")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", width=width, height=fig.layout.height or 600, scale=2)
        print(f"saved {FIG_DIR / name}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {FIG_DIR / name}.html; PNG export failed: {e}")
    fig.show()

In [57]:
# The runs compared (results/pll, pll_test_vdd.vdd_sweep) with the PLL supply of their rows to use (None: all rows,
# supply from a battery)
IREF_ORIGINAL_CSV = PLL_DIR / "pll_test_vdd_S5_20261021_181301.csv"  # IREF as before, PLL supply 0.75 V (E36312A)
IREF_BATTERY_CSV = PLL_DIR / "pll_test_vdd_S5_20261022_131911.csv"  # IREF from a battery, PLL supply 0.75 V
# IREF and PLL supply from batteries: the cleanest run before the reversed tantalums on the board were fixed (the only
# one with both settings locked and a Phase Noise mode result) and the run with new batteries as PLL supply (17:41).
# TANTALUMS_FIXED_CSV: IREF from a battery only, after the fix (PLL supply not from a battery, not recorded)
SUPPLY_BATTERY_CSV = PLL_DIR / "pll_test_vdd_S5_20261026_155550.csv"
TANTALUMS_FIXED_CSV = PLL_DIR / "pll_test_vdd_S5_20261028_165040.csv"
NEW_BATTERIES_CSV = PLL_DIR / "pll_test_vdd_S5_20261028_174157.csv"
# 16:25: the only run without the 1.65 MHz hump, but with a hump at ~3.85 MHz instead (29 ps at 1 GHz; 760 MHz not
# locked); taken while the tantalums were being fixed
HUMP_MOVED_CSV = PLL_DIR / "pll_test_vdd_S5_20261028_162507.csv"
SPECTRUM_CAPTURES = (1, 2)  # SHOWCASE_CAPTURES: 1 = +-3 MHz at 1 kHz, 2 = +-10 MHz at 3 kHz, 3 = +-1 MHz at 300 Hz
L_OFFSETS = {1000: "1k", 10000: "10k", 100000: "100k", 1000000: "1M", 10000000: "10M"}  # L_pnm_<Hz> columns


def run_label(csv_path, what):
    """'<day> <month> <hh:mm> <what>' from the stamp in the file name."""
    stamp = pd.to_datetime("_".join(csv_path.stem.split("_")[-2:]), format="%Y%m%d_%H%M%S")
    return f"{stamp:%d %b %H:%M} {what}"


def band_jitter(curve, f_pad):
    """(total, random, spurs) RMS jitter [ps] of a Phase Noise mode L(f) (on the pad, carrier `f_pad` [Hz]) over
    JITTER_BAND (pll_util.split_jitter); NaN without a curve."""
    if not isinstance(curve, pd.DataFrame):
        return (np.nan,) * 3
    c = curve[(curve["freq"] >= JITTER_BAND[0]) & (curve["freq"] <= JITTER_BAND[1])].sort_values("freq")
    j = split_jitter(f_pad, c["freq"].to_numpy(), c["level_dbm"].to_numpy())
    return j["total"] * 1e12, j["noise"] * 1e12, j["spur"] * 1e12


def load_run(csv_path, vdd):
    """The first row per setting (f_vco) of a measure_pll_vctrl CSV at PLL supply `vdd` [V] (None: all rows), with
    its Phase Noise mode L(f) in column "curve" (the k-th L_pnmode curve of the _traces.csv belongs to the k-th row
    with pnm_status ok; None without a result) and its RMS jitter over JITTER_BAND in jitter_ps / random_ps / spurs_ps
    (band_jitter)."""
    rows = pd.read_csv(csv_path)
    t = pd.read_csv(csv_path.with_name(csv_path.stem + "_traces.csv"))
    t = t[t["kind"] == "L_pnmode"]
    blocks = (t["freq"].diff() < 0).cumsum().to_numpy()
    ok = rows.index[rows["pnm_status"] == "ok"]
    n_curves = blocks.max() + 1 if len(t) else 0
    assert n_curves == len(ok), f"{csv_path.name}: {n_curves} curves for {len(ok)} rows"
    curves = {i: t[blocks == k][["freq", "level_dbm"]] for k, i in enumerate(ok)}
    rows["curve"] = [curves.get(i) for i in rows.index]
    jitter = [band_jitter(c, f / d) for c, f, d in zip(rows["curve"], rows["f_vco"], rows["total_div"])]
    for k, name in enumerate(("jitter_ps", "random_ps", "spurs_ps")):
        rows[name] = [j[k] for j in jitter]
    if vdd is not None:
        rows = rows[(rows["pll_vdd"] - vdd).abs() < 0.005]
    return rows.sort_values("time").drop_duplicates("f_vco")


def load_spectra(csv_path, vdd):
    """Captures SPECTRUM_CAPTURES of the first spectra per setting (f_vco) at PLL supply `vdd` [V] (None: any)."""
    s = pd.read_csv(csv_path.with_name(csv_path.stem + "_spectra.csv"))
    s = s[s["capture"].isin(SPECTRUM_CAPTURES)]
    if vdd is not None:
        s = s[(s["vdd_set"] - vdd).abs() < 0.005]
    return s[s["time"] == s.groupby("f_vco")["time"].transform("min")]


runs = [(run_label(IREF_ORIGINAL_CSV, "IREF original"), "#9a9893", IREF_ORIGINAL_CSV, 0.75),
        (run_label(IREF_BATTERY_CSV, "IREF battery"), "#2a78d6", IREF_BATTERY_CSV, 0.75),
        (run_label(SUPPLY_BATTERY_CSV, "IREF + supply battery"), "#eb6834", SUPPLY_BATTERY_CSV, None),
        (run_label(HUMP_MOVED_CSV, "hump at 3.9 MHz"), "#8e5bd6", HUMP_MOVED_CSV, None),
        (run_label(TANTALUMS_FIXED_CSV, "IREF battery"), "#1baf7a", TANTALUMS_FIXED_CSV, None),
        (run_label(NEW_BATTERIES_CSV, "IREF + supply battery"), "#eda100", NEW_BATTERIES_CSV, None)]
data = [(label, color, load_run(p, vdd), load_spectra(p, vdd)) for label, color, p, vdd in runs]
for label, _, p, _ in runs:
    print(f"{label}: {p.name}")

21 Oct 18:13 IREF original: pll_test_vdd_S5_20261021_181301.csv
22 Oct 13:19 IREF battery: pll_test_vdd_S5_20261022_131911.csv
26 Oct 15:55 IREF + supply battery: pll_test_vdd_S5_20261026_155550.csv
28 Oct 16:25 hump at 3.9 MHz: pll_test_vdd_S5_20261028_162507.csv
28 Oct 16:50 IREF battery: pll_test_vdd_S5_20261028_165040.csv
28 Oct 17:41 IREF + supply battery: pll_test_vdd_S5_20261028_174157.csv


In [58]:
# Per run and setting: lock, carrier, RMS jitter over JITTER_BAND (from the Phase Noise mode L(f); random / spurs) and
# L(f) at the decades referred to the VCO (+20 log10 of the output divider) [dBc/Hz]
print(f"RMS jitter (jitter_ps, random_ps, spurs_ps) integrated over {BAND_TEXT}")
table = []
for label, _, rows, _ in data:
    for _, r in rows.iterrows():
        to_vco = 20 * np.log10(r["total_div"])
        table.append(dict(f_vco_MHz=r["f_vco"] / 1e6, run=label, locked=r["locked"], carrier_dbm=r["carrier_dbm"],
                          jitter_ps=r["jitter_ps"], random_ps=r["random_ps"], spurs_ps=r["spurs_ps"],
                          **{f"L_{name}": r[f"L_pnm_{f}"] + to_vco for f, name in L_OFFSETS.items()}))
pd.DataFrame(table).sort_values(["f_vco_MHz"], kind="stable").round(1)

RMS jitter (jitter_ps, random_ps, spurs_ps) integrated over 10 kHz-10 MHz


,f_vco_MHz,run,locked,carrier_dbm,jitter_ps,random_ps,spurs_ps,L_1k,L_10k,L_100k,L_1M,L_10M
0,760.0,21 Oct 18:13 IREF original,1,0.3,357.5,357.5,0.0,-70.0,-68.9,-74.0,-68.2,-81.6
2,760.0,22 Oct 13:19 IREF battery,1,0.5,100.7,100.7,0.0,-87.4,-87.0,-78.2,-72.6,-97.0
4,760.0,26 Oct 15:55 IREF + supply battery,1,0.5,198.2,198.2,0.0,-82.9,-76.8,-75.8,-69.0,-92.6
6,760.0,28 Oct 16:25 hump at 3.9 MHz,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,760.0,28 Oct 16:50 IREF battery,1,0.6,95.2,95.2,0.0,-87.4,-86.5,-78.8,-72.5,-97.6
10,760.0,28 Oct 17:41 IREF + supply battery,1,0.5,83.8,83.8,0.0,-89.1,-88.4,-79.7,-74.8,-97.8
1,1000.0,21 Oct 18:13 IREF original,1,1.6,45.6,44.2,11.3,-91.8,-95.2,-98.4,-92.9,-91.7
3,1000.0,22 Oct 13:19 IREF battery,1,1.5,12.3,12.3,0.0,-91.1,-96.5,-100.8,-95.2,-107.0
5,1000.0,26 Oct 15:55 IREF + supply battery,1,1.6,18.6,18.3,3.2,-93.4,-97.2,-100.3,-97.3,-103.5
7,1000.0,28 Oct 16:25 hump at 3.9 MHz,1,1.6,29.0,29.0,1.3,-92.9,-97.8,-101.9,-99.3,-108.6


In [59]:
# Per setting: the Phase Noise mode L(f), the RMS jitter and the spectrum around the carrier of every run
for f_vco in sorted(set().union(*(rows["f_vco"] for _, _, rows, _ in data))):
    tag, mhz = f"{f_vco / 1e6:.0f}MHz", f"{f_vco / 1e6:.0f} MHz"
    here = [(label, color, rows[rows["f_vco"] == f_vco].iloc[0], s[s["f_vco"] == f_vco].sort_values("freq"))
            for label, color, rows, s in data if (rows["f_vco"] == f_vco).any()]

    fig = go.Figure()
    for label, color, r, _ in here:
        if isinstance(r["curve"], pd.DataFrame):
            t = binned(r["curve"], F_START, F_STOP)
            fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines", name=label,
                            line=dict(width=2, color=color),
                            hovertemplate=label + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Phase noise, battery tests, {mhz} (dotted: jitter band {BAND_TEXT})", height=550,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS, range=[np.log10(F_START), np.log10(F_STOP)])
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    for f in JITTER_BAND:
        fig.add_vline(x=np.log10(f), line=dict(width=1, dash="dot", color="#52514e"))  # log axis: shapes in log10
    save(fig, f"phase_noise_battery_{tag}")

    fig = go.Figure()
    fig.add_bar(x=[label + (" (not locked)" if r["locked"] != 1 else " (no result)"
                            if pd.isna(r["jitter_ps"]) else "") for label, _, r, _ in here],
                y=[r["jitter_ps"] for _, _, r, _ in here], marker_color=[color for _, color, _, _ in here],
                text=[f"{r['random_ps']:.0f} random<br>{r['spurs_ps']:.0f} spurs" for _, _, r, _ in here],
                hovertemplate="%{x}: %{y:.1f} ps<extra></extra>")
    fig.update_layout(title=f"RMS jitter {BAND_TEXT}, battery tests, {mhz}", height=600)
    fig.update_xaxes(gridcolor=GRID)
    fig.update_yaxes(title=f"RMS jitter {BAND_TEXT} [ps]", gridcolor=GRID, rangemode="tozero")
    save(fig, f"jitter_battery_{tag}")

    for capture in SPECTRUM_CAPTURES:
        fig, rbw, span = go.Figure(), None, None
        for label, color, _, s in here:
            s = s[s["capture"] == capture]
            if len(s) and s["locked"].iloc[0] == 1:
                rbw, span = s["rbw"].iloc[0], s["span"].iloc[0]
                fig.add_scatter(x=(s["freq"] - s["f_carrier"]) / 1e6,
                                y=s["level_dbm"] - s["carrier_dbm"] + 20 * np.log10(s["total_div"]), mode="lines",
                                name=label, line=dict(width=1, color=color),
                                hovertemplate=label + "<br>%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
        if rbw is not None:
            fig.update_layout(title=f"Spectrum around the carrier (+-{span / 2e6:g} MHz), battery tests, {mhz}",
                              height=550, legend=dict(orientation="h", y=-0.2))
            fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
            fig.update_yaxes(title=f"level referred to the VCO [dBc in {rbw:g} Hz RBW]", gridcolor=GRID)
            save(fig, f"spectrum_battery_{tag}_{span / 2e6:g}MHz")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/phase_noise_battery_760MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/jitter_battery_760MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_760MHz_3MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_760MHz_10MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/phase_noise_battery_1000MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/jitter_battery_1000MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_1000MHz_3MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_1000MHz_10MHz.html and .png


In [60]:
# The widest spectra per setting: the 42 MHz span of the full measurement (measure_pll_vctrl: phase_noise trace in the
# _traces.csv, +-21 MHz around the carrier at 50 kHz RBW, the first at the run's PLL supply) and, in the runs that have
# it, the wide capture of the spectra (pll_test_vdd.WIDE_CAPTURES: 1 MHz - 1.5 GHz, absolute frequency, dBc on the pad)
WIDE_TRACE_SPAN = 42e6  # span of the widest phase_noise trace [Hz]
WIDE_CAPTURE = 4  # capture number of the wide spectrum in the _spectra.csv


def widest_traces(csv_path, vdd):
    """Per setting (f_vco) the first WIDE_TRACE_SPAN phase_noise trace of a run's _traces.csv at PLL supply `vdd` [V]
    (None: any): {f_vco: DataFrame (freq, level_dbm, center, rbw, ...)}."""
    t = pd.read_csv(csv_path.with_name(csv_path.stem + "_traces.csv"))
    t = t[(t["kind"] == "phase_noise") & (t["span"] == WIDE_TRACE_SPAN)]
    if vdd is not None:
        t = t[(t["pll_vdd"] - vdd).abs() < 0.005]
    # each trace sweeps upwards: a new one starts where the frequency drops
    return {f: g[(g["freq"].diff() < 0).cumsum() == 0] for f, g in t.groupby("f_vco")}


def wide_spectra(csv_path, vdd):
    """The first WIDE_CAPTURE spectrum per setting of a run's _spectra.csv at PLL supply `vdd` [V] (None: any); empty
    for runs from before pll_test_vdd.WIDE_CAPTURES."""
    s = pd.read_csv(csv_path.with_name(csv_path.stem + "_spectra.csv"))
    s = s[s["capture"] == WIDE_CAPTURE]
    if vdd is not None:
        s = s[(s["vdd_set"] - vdd).abs() < 0.005]
    return s[s["time"] == s.groupby("f_vco")["time"].transform("min")]


wide = [(widest_traces(p, vdd), wide_spectra(p, vdd)) for _, _, p, vdd in runs]
for f_vco in sorted(set().union(*(rows["f_vco"] for _, _, rows, _ in data))):
    tag, mhz = f"{f_vco / 1e6:.0f}MHz", f"{f_vco / 1e6:.0f} MHz"

    fig, rbw = go.Figure(), None
    for (label, color, rows, _), (traces, _) in zip(data, wide):
        r = rows[rows["f_vco"] == f_vco]
        if f_vco not in traces or not len(traces[f_vco]) or not len(r) or r["locked"].iloc[0] != 1:
            continue
        t, r = traces[f_vco], r.iloc[0]
        rbw = t["rbw"].iloc[0]
        fig.add_scatter(x=(t["freq"] - t["center"]) / 1e6,
                        y=t["level_dbm"] - r["carrier_dbm"] + 20 * np.log10(r["total_div"]), mode="lines", name=label,
                        line=dict(width=1, color=color),
                        hovertemplate=label + "<br>%{x:.3f} MHz: %{y:.1f} dBc<extra></extra>")
    if rbw is not None:
        fig.update_layout(title=f"Spectrum around the carrier (+-{WIDE_TRACE_SPAN / 2e6:g} MHz), battery tests, {mhz}",
                          height=550, legend=dict(orientation="h", y=-0.2))
        fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
        fig.update_yaxes(title=f"level referred to the VCO [dBc in {rbw / 1e3:g} kHz RBW]", gridcolor=GRID)
        save(fig, f"spectrum_battery_{tag}_{WIDE_TRACE_SPAN / 2e6:g}MHz")

    fig, rbw = go.Figure(), None
    for (label, color, _, _), (_, s) in zip(data, wide):
        s = s[s["f_vco"] == f_vco].sort_values("freq")
        if len(s):
            rbw, f_pad, f_ref = s["rbw"].iloc[0], s["f_pad"].iloc[0], s["f_ref"].iloc[0]
            fig.add_scatter(x=s["freq"] / 1e6, y=s["level_dbm"] - s["carrier_dbm"], mode="lines", name=label,
                            line=dict(width=1, color=color),
                            hovertemplate=label + "<br>%{x:.3f} MHz: %{y:.1f} dBc<extra></extra>")
    if rbw is None:
        print(f"{mhz}: no wide capture yet (runs from before pll_test_vdd.WIDE_CAPTURES)")
        continue
    for f, name in ((f_pad, "carrier"), (f_ref, "reference"), (f_vco, "VCO")):
        fig.add_vline(x=f / 1e6, line=dict(width=1, dash="dot", color="#52514e"), annotation_text=name)
    fig.update_layout(title=f"Wide spectrum, battery tests, {mhz} (carrier {f_pad / 1e6:g} MHz on the pad)",
                      height=550, legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="frequency [MHz]", gridcolor=GRID)
    fig.update_yaxes(title=f"level re. the carrier [dBc in {rbw / 1e3:g} kHz RBW]", gridcolor=GRID)
    save(fig, f"spectrum_battery_{tag}_wide")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_760MHz_21MHz.html and .png


760 MHz: no wide capture yet (runs from before pll_test_vdd.WIDE_CAPTURES)
saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/spectrum_battery_1000MHz_21MHz.html and .png


1000 MHz: no wide capture yet (runs from before pll_test_vdd.WIDE_CAPTURES)


In [61]:
# Where the jitter sits: the RMS jitter integrated from JITTER_BAND[0] up to every offset (Phase Noise mode L(f), as
# band_jitter), total and random part (pll_util.split_jitter: spurs replaced by the noise under them). Spurs show as
# steps, noise as slopes; dotted: where 50 % and 90 % of the jitter power (ps^2) is reached.
# CUMULATIVE: (run CSV, f_vco) per plot; the 11 ps at 1 GHz of 28 Oct 16:50 and the 29 ps of 16:25 (hump at 3.9 MHz)
CUMULATIVE = [(TANTALUMS_FIXED_CSV, 1e9), (HUMP_MOVED_CSV, 1e9)]
DECADES = (1e4, 1e5, 1e6, 1e7)  # edges of the per-decade breakdown [Hz]


def cumulative_jitter(fc, freq, level_dbm):
    """RMS jitter [ps] on a carrier `fc` [Hz] integrated from freq[0] to every freq (as
    pll_util.integrate_phase_noise: phase variance = 2 x trapezoid integral of 10^(L/10) df)."""
    lin = 10.0 ** (np.asarray(level_dbm) / 10.0)
    var = 2.0 * np.concatenate([[0.0], np.cumsum((lin[1:] + lin[:-1]) / 2 * np.diff(freq))])
    return np.sqrt(var) / (2 * np.pi * fc) * 1e12


for csv_path, f_vco in CUMULATIVE:
    label, color, rows, _ = next(d for d, run in zip(data, runs) if run[2] == csv_path)
    tag, mhz = f"{f_vco / 1e6:.0f}MHz", f"{f_vco / 1e6:.0f} MHz"
    r = rows[rows["f_vco"] == f_vco].iloc[0]
    f_pad, c = r["f_vco"] / r["total_div"], r["curve"]
    c = c[(c["freq"] >= JITTER_BAND[0]) & (c["freq"] <= JITTER_BAND[1])].sort_values("freq")
    freq, level = c["freq"].to_numpy(), c["level_dbm"].to_numpy()
    total = cumulative_jitter(f_pad, freq, level)
    random = cumulative_jitter(f_pad, freq, split_jitter(f_pad, freq, level)["L_noise"])
    share = total ** 2 / total[-1] ** 2
    print(f"{label}, {mhz}: {total[-1]:.1f} ps over {BAND_TEXT} (band_jitter: {r['jitter_ps']:.1f} ps), "
          f"random {random[-1]:.1f} ps")
    for lo, hi in zip(DECADES[:-1], DECADES[1:]):
        part = np.sqrt(max(np.interp(hi, freq, total) ** 2 - np.interp(lo, freq, total) ** 2, 0.0))
        power = np.interp(hi, freq, share) - np.interp(lo, freq, share)
        print(f"  {lo / 1e3:>6g} - {hi / 1e3:<6g} kHz: {part:5.1f} ps, {power:5.1%} of the jitter power")

    fig = go.Figure()
    fig.add_scatter(x=freq, y=total, mode="lines", name="integrated RMS jitter, total", line=dict(width=3, color=color),
                    hovertemplate="up to %{x:.3s}Hz: %{y:.2f} ps<extra></extra>")
    fig.add_scatter(x=freq, y=random, mode="lines", name="integrated RMS jitter, random part",
                    line=dict(width=2, dash="dash", color=color),
                    hovertemplate="up to %{x:.3s}Hz: %{y:.2f} ps<extra></extra>")
    t = binned(c, *JITTER_BAND)
    fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines", yaxis="y2",
                    name="L(f) referred to the VCO", line=dict(width=1, color="#9a9893"),
                    hovertemplate="%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    # for q in (0.5, 0.9):
    #     f = freq[min(np.searchsorted(share, q), len(freq) - 1)]
    #     fig.add_vline(x=np.log10(f), line=dict(width=1, dash="dot", color="#52514e"),  # log axis: shapes in log10
    #                   annotation_text=f"{q:.0%} at {f / 1e3:.3g} kHz")
    fig.update_layout(title=f"Integrated RMS jitter from {JITTER_BAND[0] / 1e3:g} kHz, {label}, {mhz}", height=600,
                      legend=dict(orientation="h", y=-0.2),
                      yaxis2=dict(title="L(f) referred to the VCO [dBc/Hz]", overlaying="y", side="right",
                                  showgrid=False))
    fig.update_xaxes(title="offset from the carrier (integration upper limit)", **OFFSET_AXIS,
                     range=[np.log10(JITTER_BAND[0]), np.log10(JITTER_BAND[1])])
    fig.update_layout(yaxis=dict(title="RMS jitter integrated up to the offset [ps]", gridcolor=GRID,
                                 rangemode="tozero"))
    save(fig, f"cumulative_jitter_{csv_path.stem.split('_', 4)[-1]}_{tag}")

28 Oct 16:50 IREF battery, 1000 MHz: 11.0 ps over 10 kHz-10 MHz (band_jitter: 11.0 ps), random 11.0 ps
      10 - 100    kHz:   0.7 ps,  0.4% of the jitter power
     100 - 1000   kHz:   2.9 ps,  7.1% of the jitter power
    1000 - 10000  kHz:  10.6 ps, 92.5% of the jitter power
saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/cumulative_jitter_20261028_165040_1000MHz.html and .png


28 Oct 16:25 hump at 3.9 MHz, 1000 MHz: 29.0 ps over 10 kHz-10 MHz (band_jitter: 29.0 ps), random 29.0 ps
      10 - 100    kHz:   0.6 ps,  0.0% of the jitter power
     100 - 1000   kHz:   1.8 ps,  0.4% of the jitter power
    1000 - 10000  kHz:  29.0 ps, 99.6% of the jitter power
saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/cumulative_jitter_20261028_162507_1000MHz.html and .png


## The good run alone: 28 Oct 16:50, 1 GHz (11 ps) and 760 MHz

Every spectrum of each setting of `GOOD_RUNS` on its own: the showcase captures of the `_spectra.csv` (+-3 / +-10 / +-1 MHz, and the 1 MHz-1.5 GHz capture when there is one), then the traces of the full measurement in the `_traces.csv` (the zooms from 21 kHz to 42 MHz span and the 5 MHz spectrum).

In [62]:
# The good run alone: every spectrum of each (run CSV, f_vco) of GOOD_RUNS on its own, levels referred to the VCO
# (+20 log10 of the output divider) except the wide capture (dBc on the pad, absolute frequency)
GOOD_RUNS = [(TANTALUMS_FIXED_CSV, 1e9), (TANTALUMS_FIXED_CSV, 760e6)]


def good_setting(csv_path, f_vco):
    """dict(csv, f_vco, label, color, row: its load_run row, tag, title, to_vco [dB]) of one entry of GOOD_RUNS."""
    label, color, rows, _ = next(d for d, run in zip(data, runs) if run[2] == csv_path)
    row = rows[rows["f_vco"] == f_vco].iloc[0]
    return dict(csv=csv_path, f_vco=f_vco, label=label, color=color, row=row,
                tag=f"{csv_path.stem.split('_', 4)[-1]}_{f_vco / 1e6:.0f}MHz", title=f"{label}, {f_vco / 1e6:.0f} MHz",
                to_vco=20 * np.log10(row["total_div"]))


def offset_axis(span):
    """(scale, unit) of an offset axis for a span [Hz]: kHz below 1 MHz, MHz above."""
    return (1e3, "kHz") if span < 1e6 else (1e6, "MHz")


def plot_good(g, freq, center, level_dbc, span, rbw, what, name):
    """One spectrum of setting `g` (good_setting) against the offset from `center` [Hz]; `level_dbc` already referred
    to the VCO."""
    scale, unit = offset_axis(span)
    fig = go.Figure()
    fig.add_scatter(x=(freq - center) / scale, y=level_dbc, mode="lines", name=g["label"],
                    line=dict(width=1, color=g["color"]),
                    hovertemplate="%{x:.4f} " + unit + ": %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"{what} (+-{span / 2 / scale:g} {unit}, RBW {rbw:g} Hz), {g['title']}", height=550)
    fig.update_xaxes(title=f"offset from the carrier [{unit}]", gridcolor=GRID)
    fig.update_yaxes(title=f"level referred to the VCO [dBc in {rbw:g} Hz RBW]", gridcolor=GRID)
    save(fig, f"good_{g['tag']}_{name}")


def plot_good_spectra(g):
    """The showcase captures (_spectra.csv) of setting `g` (good_setting), the first of the setting, one plot each."""
    s = pd.read_csv(g["csv"].with_name(g["csv"].stem + "_spectra.csv"))
    s = s[s["f_vco"] == g["f_vco"]]
    s = s[s["time"] == s["time"].min()]
    for capture, c in s.groupby("capture"):
        c = c.sort_values("freq")
        span, rbw = c["span"].iloc[0], c["rbw"].iloc[0]
        if capture != WIDE_CAPTURE:
            plot_good(g, c["freq"], c["f_carrier"].iloc[0], c["level_dbm"] - c["carrier_dbm"] + g["to_vco"], span, rbw,
                      f"Spectrum, capture {capture}", f"capture{capture}")
            continue
        fig = go.Figure()
        fig.add_scatter(x=c["freq"] / 1e6, y=c["level_dbm"] - c["carrier_dbm"], mode="lines", name=g["label"],
                        line=dict(width=1, color=g["color"]),
                        hovertemplate="%{x:.3f} MHz: %{y:.1f} dBc<extra></extra>")
        for f, name in ((c["f_pad"].iloc[0], "carrier"), (c["f_ref"].iloc[0], "reference"), (g["f_vco"], "VCO")):
            fig.add_vline(x=f / 1e6, line=dict(width=1, dash="dot", color="#52514e"), annotation_text=name)
        fig.update_layout(title=f"Wide spectrum (RBW {rbw / 1e3:g} kHz), {g['title']}", height=550)
        fig.update_xaxes(title="frequency [MHz]", gridcolor=GRID)
        fig.update_yaxes(title=f"level re. the carrier [dBc in {rbw / 1e3:g} kHz RBW]", gridcolor=GRID)
        save(fig, f"good_{g['tag']}_capture{capture}_wide")


for csv_path, f_vco in GOOD_RUNS:
    plot_good_spectra(good_setting(csv_path, f_vco))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_capture1.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_capture2.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_capture3.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_capture1.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_capture2.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_capture3.html and .png


In [63]:
# The traces of the full measurement (_traces.csv) per entry of GOOD_RUNS: the phase_noise zooms (21 kHz ... 42 MHz
# span) and the 5 MHz spectrum, the first of each (kind, span) of the setting; dBc against the carrier of the
# measurement (carrier_dbm)
def plot_good_traces(g):
    """The full-measurement traces of setting `g` (good_setting), one plot each."""
    t = pd.read_csv(g["csv"].with_name(g["csv"].stem + "_traces.csv"))
    t = t[(t["f_vco"] == g["f_vco"]) & t["kind"].isin(["phase_noise", "spectrum"])]
    for (kind, span), tr in t.groupby(["kind", "span"]):
        tr = tr[(tr["freq"].diff() < 0).cumsum() == 0]  # the first trace (each one sweeps upwards)
        plot_good(g, tr["freq"], tr["center"].iloc[0], tr["level_dbm"] - g["row"]["carrier_dbm"] + g["to_vco"], span,
                  tr["rbw"].iloc[0], "Full measurement, " + kind.replace("_", " ") + " trace",
                  f"{kind}_{span / 1e3:g}kHz")


for csv_path, f_vco in GOOD_RUNS:
    plot_good_traces(good_setting(csv_path, f_vco))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_phase_noise_21kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_phase_noise_210kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_phase_noise_2100kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_phase_noise_21000kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_phase_noise_42000kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_1000MHz_spectrum_5000kHz.html and .png


... (2 outputs left out)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_phase_noise_2100kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_phase_noise_21000kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_phase_noise_42000kHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_battery/good_20261028_165040_760MHz_spectrum_5000kHz.html and .png
